# load_openalex_institution

Prototipo del nodo `load_openalex_institution` del pipeline `load_openalex`. No guarda datasets.


In [ ]:
import pandas as pd
from pandas import json_normalize

%load_ext kedro.ipython


In [ ]:
df_institution_raw = catalog.load('raw/openalex/institution/parquet/institution_dev')
df_institution_raw.head(2)


In [ ]:
def _select_with_metadata(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    df = _add_openalex_extracted_metadata(df)
    return df.loc[:, [*columns, *_EXTRACTED_META_COLS]].copy()


In [ ]:
def _stringify_object_columns(
    df: pd.DataFrame,
    exclude_columns: list[str] | None = None,
) -> pd.DataFrame:
    exclude_columns = set(exclude_columns or [])
    for column in df.columns:
        if column in exclude_columns:
            continue
        if pd.api.types.is_object_dtype(df[column]):
            df[column] = df[column].where(df[column].notna(), pd.NA).astype("string")
    return df


In [ ]:
def _serialize_nested_value(value):
    if value is None or value is pd.NA:
        return value
    if hasattr(value, "tolist") and not isinstance(value, (str, bytes)):
        value = value.tolist()
    if isinstance(value, (dict, list, tuple, set)):
        return json.dumps(value, ensure_ascii=False, default=str)
    return value


In [ ]:
def _serialize_nested_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for column in df.columns:
        if pd.api.types.is_object_dtype(df[column]):
            df[column] = df[column].map(_serialize_nested_value)
    return df


In [ ]:
def _add_openalex_extracted_metadata(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in _EXTRACTED_META_COLS:
        if col not in df.columns:
            df[col] = pd.NA
    df["extract_datetime"] = pd.to_datetime(df["extract_datetime"], errors="coerce")
    df["_extract_datetime"] = pd.to_datetime(df["_extract_datetime"], errors="coerce")
    if "extract_date" in df.columns:
        df["extract_date"] = pd.to_datetime(df["extract_date"], errors="coerce").dt.date
    return df


In [ ]:
def _add_openalex_loaded_metadata(df: pd.DataFrame, load_datetime=None) -> pd.DataFrame:
    df = _serialize_nested_columns(df)
    if load_datetime is None:
        load_datetime = pd.Timestamp.now(tz="UTC").floor("s").tz_localize(None)
    load_datetime = pd.to_datetime(load_datetime)
    df["_load_datetime"] = load_datetime
    return df


In [ ]:
def load_openalex_institution(df_institution_raw):

    df_institution_raw = _add_openalex_extracted_metadata(df_institution_raw)

    expected_columns = [
        'id',
        'ror',
        'display_name',
        'country_code',
        'type',
        'type_id',
        'lineage',
        'homepage_url',
        'image_url',
        'image_thumbnail_url',
        'display_name_acronyms',
        'display_name_alternatives',
        'repositories',
        'works_count',
        'cited_by_count',
        'summary_stats',
        'ids',
        'geo',
        'international',
        'associated_institutions',
        'counts_by_year',
        'roles',
        'topics',
        'topic_share',
        'is_super_system',
        'works_api_url',
        'updated_date',
        'created_date',
        *_CORE_EXTRACTED_META_COLS,
        '_filter_param',
        '_filter_value',
        '_extract_datetime',
    ]
    df_institution = df_institution_raw.reindex(columns=expected_columns).reset_index(drop=True).copy()

    df_institution = _add_openalex_loaded_metadata(df_institution)

    df_institution = _stringify_object_columns(
        df_institution,
        exclude_columns=[
            "extract_datetime",
            "_extract_datetime",
            "extract_date",
            "_load_datetime",
        ],
    )
    df_institution = df_institution.convert_dtypes()

    return df_institution


In [ ]:
df_institution = load_openalex_institution(df_institution_raw)


In [ ]:
pd.DataFrame([{'dataset': 'df_institution', 'rows': len(df_institution), 'columns': len(df_institution.columns)}])


In [ ]:
df_institution.head(2)
